# **Introduction to Pandas, Part 2: Exercises**
---
Exercises to accompany `Session_5_IntroToPandasPart2.ipynb`. The problems follow the order of the notebook:

1. Building and indexing a DataFrame
2. Modifying DataFrames, `apply`, and copies
3. Returns and moving averages on real data
4. `groupby` on the class grades
5. `groupby` and reshaping: the chopstick experiment
6. Merging and concatenating: Amazon options
7. Data cleaning and filling gaps: EUR/GBP ticks
8. Challenge: combining series and building a volatility signal

The data files are the ones in the course repository, loaded directly from GitHub. Problems 3 and 8 download prices with `yfinance`, as in the notebook.

Before running anything, try to **predict** what each command will return. Most of the learning happens when the prediction turns out wrong.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA_URL = "https://github.com/tdhadfield/MATH_70112/raw/main/"

# Exercise 1: Building and indexing a DataFrame

**(a)** Fix the random seed with `np.random.seed(42)`. Create a DataFrame `df` of shape (500, 4) of standard Gaussian values with columns `"A"`, `"B"`, `"C"`, `"D"`, indexed by **business days** starting on 2024-01-01 (hint: the `freq` argument of `pd.date_range`). Name the index `"Date"`.

Print its shape, the first and last dates, and `df.describe()`.

**(b)** Retrieve the value in the 10th row of column `"C"` in three different ways: with `.iloc`, with `.loc`, and with `.iat`. Check that they agree.

**(c)** Using `.loc` with date strings, select columns `"A"` and `"B"` for all of March 2024. How many rows are there, and why is it not 31?

**(d)** How many rows have `A > 2` **or** `B < -2`? How many have both? Why are the parentheses needed in `(df["A"] > 2) | (df["B"] < -2)`?

**(e)** For each column, find the date on which its maximum is attained, in a single line of code.

**(f)** Predict the type (and, where relevant, the shape) of each of the following, then check:

- `df["A"]`
- `df[["A"]]`
- `df.iloc[0]`
- `df.iloc[0:1]`
- `df.loc["2024-01-02", "A"]`
- `df["A"].values`

# Exercise 2: Modifying DataFrames, `apply`, and copies

Keep working with `df` from Problem 1.

**(a)** In the notebook, the cell calling `df.rename(columns={...})` did not actually rename the columns of `df`. Explain why, and rename `A, B, C, D` to `SPX, DAX, GOOGL, AAPL` so that the change sticks.

**(b)** Set `DAX` to 0 on every row where `SPX > 2`.

First try the "chained" version `df[df["SPX"] > 2]["DAX"] = 0.` and check whether `df` changed. Then do it correctly with `.loc`. Explain the difference.

**(c)** Run the code below, but first predict the three printed values.

In [ ]:
small = pd.DataFrame(np.zeros((3, 2)), columns=["x", "y"])
s1 = small                     # plain assignment
s2 = small.copy(deep=False)    # shallow copy
s3 = small.copy()              # deep copy (the default)

print(small)

small.iat[0, 0] = 99.
print(s1.iat[0, 0], s2.iat[0, 0], s3.iat[0, 0])
print(small)

Explain the result. In particular, is `s1 = small` the same as `small.copy(deep=False)`?

**(d)** Write a function `cap(x, level)` that returns `x` clipped to the interval `[-level, level]`, and use `.apply` with the `args` argument to create a column `"SPX_capped"` with `level=1.5`.

Then do the same thing with the built-in `Series.clip` method and check the results are identical. Use `%timeit` to compare the two on a Series of 1,000,000 random values.

**(e)** The notebook asks *"What is wrong with the plot?"* after `df.plot()`. Plot the cumulative sum of each of the four original series instead (think of them as daily returns), and explain why this plot is more informative.

# Exercise 3: Returns and moving averages

**(a)** Download daily closing prices of the S&P 500 (`"^GSPC"`) from 2010-01-01 to 2025-01-01 with `yfinance`, as a Series `px`.

*Note:* recent versions of `yfinance` return columns with two levels, e.g. `("Close", "^GSPC")`, so `yf.download(...)["Close"]` is a one-column DataFrame. `.squeeze()` turns it into a Series.

In [ ]:
import yfinance as yf

px = yf.download("^GSPC", start="2010-01-01", end="2025-01-01", auto_adjust=True, progress=False)["Close"].squeeze()
px.name = "SPX"
px.tail()

**(b)** Build a DataFrame `spx` with columns `Close`, `LogRet` (log returns using `shift`) and `SimpleRet` (using `pct_change`). Drop the missing value(s). What is the largest absolute difference between the two kinds of return, and on which date does it occur? Why there?

**(c)** Using `groupby` on the year of the index (`spx.index.year`), compute for each year:

- the annualised volatility of log returns (std × √252),
- the total log return over the year,
- the number of trading days.

Which year was the most volatile?

**(d)** Add the 50-day and 200-day moving averages of the close, `MA50` and `MA200`. Create a column `Signal` equal to 1 when `MA50 > MA200` and 0 otherwise. Plot `Close`, `MA50` and `MA200` together, and report the fraction of days with `Signal == 1`.

*What happens to `Signal` during the first 200 days? Is that what you want?*

**(e)** Trade the signal: hold the index when the signal is 1 and cash (zero return) when it is 0. The strategy's daily log return is `Signal × LogRet`, **but** the signal computed with today's close can only be acted on tomorrow.

Compute the strategy return correctly using `shift`, and compare the cumulative log return with buy-and-hold over the period where the signal exists. Then repeat it *without* the shift. Why does this look-ahead bias flatter the strategy?

How does this compare to simply buying and holding the index?

**(f)** Compute the 60-day rolling annualised volatility `Vol60` of log returns. Show the five days with the largest absolute log returns together with the value of `Vol60` on each of those days. What do you notice? (Recall the "volatility clustering" remark in the notebook.)

# Exercise 4: `groupby` on the class grades

Load `Class.xlsx` as in the notebook.

In [ ]:
# !pip install openpyxl

In [ ]:
file_name = DATA_URL + "Class.xlsx"
#"https://github.com/tdhadfield/MATH_70112/raw/main/Class.xlsx"

grades = pd.read_excel(file_name)
print(grades.shape)
print(grades.tail())

### Note: this is completely random data!

**(a)** For each nationality, compute the number of students, the mean grade and the highest grade in one call to `.agg`. Sort by mean grade, highest first.

**(b)** Compute the mean grade by nationality *and* gender, then use `.unstack()` to show nationalities as rows and genders as columns. Why do some cells contain `NaN`? (This is the *"What is happening?"* question in the notebook.)

**(c)** Without using a loop, add a column `"vs_nat_mean"` equal to each student's grade minus the mean grade of their nationality (hint: `transform`). Who is furthest above their nationality's average?

**(d)** Find the top student of each nationality (name and grade).

**(e)** Add a boolean column `"Pass"` (grade at least 70) and compute the pass rate by gender. Why does taking the mean of a boolean column give a rate?

# Exercise 5: `groupby` and reshaping: the chopstick experiment

`chopstick.csv` records the food-pinching efficiency of 31 individuals, each tested with chopsticks of six different lengths (in mm).

In [ ]:
chop = pd.read_csv(DATA_URL + "chopstick.csv")
chop.head()

**(a)** Rename the columns to `"Efficiency"`, `"Individual"`, `"Length"`. Check that every individual was tested with every length exactly once.

**(b)** Compute the mean and standard deviation of efficiency for each length and plot the means as a bar chart. Which length looks best on average?

**(c)** Reshape the data with `pivot` so that each row is an individual and each column a length. For each individual, find the length at which they were most efficient, and count how many individuals did best with each length. Does this agree with (b)?

# Exercise 6: Merging and concatenating: Amazon options

`AMZN_options.csv` contains call (`C`) and put (`P`) quotes on Amazon for several expiries.

In [ ]:
opts = pd.read_csv(DATA_URL + "AMZN_options.csv")
opts[["strike", "bid", "ask", "volume", "expiration_date", "option_type"]].head()

**(a)** For each expiration date, compute the number of contracts and the total traded volume. Which expiry is most traded?

**(b)** Add a column `"mid"` = (bid + ask)/2. Split the data into `calls` and `puts`, each keeping only `expiration_date`, `strike` and `mid`. Merge them on expiry **and** strike so that each row holds a call and a put with the same expiry and strike, with columns `mid_C` and `mid_P`.

**(c)** Repeat the merge with `how="outer"` and `indicator=True`. How many strikes have only a call, and how many only a put? Why are the inner and outer merges different sizes?

**(d)** Use `pd.concat` to stack `calls` and `puts` back into a single DataFrame with an extra column saying which is which (hint: the `keys` argument, followed by `reset_index`). Check that the number of rows equals the original.

# Exercise 7: Data cleaning and filling gaps

**(a)** Messy data. Clean the DataFrame below so that `price` and `qty` are numeric, unknown values are `NaN`, and `-1` (used as a placeholder for "missing" in `qty`) is also `NaN`. Then compute the total value `price × qty`, ignoring missing rows.

**(b)** EUR/GBP ticks. The file `EURGBP-2020-01.csv` has **no header row** and four columns: the currency pair, a timestamp such as `20200101 22:00:32.963`, the bid and the ask.

Load it with suitable column names, convert the timestamp to a datetime (look at the `format` argument of `pd.to_datetime`), make it the index, and drop the currency-pair column. Add columns `Mid` and `Spread`. How many ticks are there, and over what time span?

In [ ]:
file_name = DATA_URL + "EURGBP-2020-01.csv"
fx = pd.read_csv(file_name)

**(c)** Compute the number of ticks and the last mid price in each one-minute interval (hint: `resample("1min")`).

**(d)** Filling the void. Make a copy of the `Bid` series and, using `np.random.seed(0)`, replace a random 20% of its values with `NaN`. Fill the gaps in four ways:

1. forward fill (`ffill`),
2. linear interpolation (`interpolate(method="linear")`),
3. time-weighted interpolation (`interpolate(method="time")`),
4. dropping them (`dropna`).

For methods 1 to 3, compute the root-mean-square error against the true values at the missing points. Which method works best here, and why might `"time"` differ from `"linear"` for tick data?

# Exercise 8 (challenge): Combining series and a volatility signal

This extends the "Extra" section of the notebook.

**(a)** Download adjusted closes for Coca-Cola (`"KO"`) and Apple (`"AAPL"`) from 2005-01-01 to 2025-01-01 and combine them into a DataFrame `prices` with columns `"KO"` and `"AAPL"`, keeping only dates where both exist. Plot both series normalised to 1 on the first date. Why is normalising necessary for comparison?

**(b)** Compute daily log returns of both stocks and their 60-day rolling correlation. Plot it. When was the correlation highest?

**(c)** Volatility targeting. For Apple, compute the 60-day rolling annualised volatility of log returns. A volatility-targeting strategy holds a position

$$w_t = \min\left(1, \frac{\sigma_{\text{target}}}{\hat\sigma_{t-1}}\right), \qquad \sigma_{\text{target}} = 20\%,$$

where $\hat\sigma_{t-1}$ is yesterday's rolling volatility, and earns $w_t \times r_t$. Compute the strategy's cumulative log return, its annualised volatility and its annualised mean return, and compare with buy-and-hold over the same dates.